In [ ]:
import rasterio
from rasterio.mask import mask
from rasterio.warp import reproject, Resampling
import numpy as np
import geopandas as gpd
import pandas as pd

from rasterstats import zonal_stats
from scipy import ndimage
from sklearn.preprocessing import StandardScaler
from shapely.geometry import mapping

import os
from pathlib import Path

In [ ]:
# ==============================
# PATH CONFIGURATION (Centralized)
# ==============================

DATA_ROOT = "SuDS_Data_Raw"

PATHS = {
    "dem": f"{DATA_ROOT}/DEM/DEM_complete.tif",
    "landuse": f"{DATA_ROOT}/Landuse/New_chelmsford_worldcover_bng.tif",
    
    "river_flood": f"{DATA_ROOT}/Risk of Flooding from Rivers and Sea/rofrs_4bandPolygon.shp",
    "surface_flood": f"{DATA_ROOT}/Risk of Flooding from Surface Water/rofswPolygon.shp",
    
    "rivers": f"{DATA_ROOT}/OS Open Rivers/data/WatercourseLink.shp",
    "roads": f"{DATA_ROOT}/Road Network/oproad_essh_gb/data/combined_road_data.gpkg",
    
    "ndvi": f"{DATA_ROOT}/SatImg/chel_202410-202505/S2_median_20241027_NDVI.tif",
    "ndwi": f"{DATA_ROOT}/SatImg/chel_202410-202505/S2_median_20241027_NDWI.tif",
    "ndbi": f"{DATA_ROOT}/SatImg/chel_202410-202505/S2_median_20241027_NDBI.tif",
    
    "study_area": f"{DATA_ROOT}/Study Boundary/Final Boundary layer.shp"
}

# ==============================
# LOAD VECTOR DATA (once only)
# ==============================

river_flood = gpd.read_file(PATHS["river_flood"])
surface_flood = gpd.read_file(PATHS["surface_flood"])
rivers = gpd.read_file(PATHS["rivers"])
roads = gpd.read_file(PATHS["roads"], layer="road_links")
study_area = gpd.read_file(PATHS["study_area"])

# ==============================
# RASTER PATHS (not loaded yet)
# ==============================

dem_path = PATHS["dem"]
landuse_path = PATHS["landuse"]

sat_NDVI = PATHS["ndvi"]
sat_NDWI = PATHS["ndwi"]
sat_NDBI = PATHS["ndbi"]

In [ ]:
import rasterio

# ==============================
# GET TARGET CRS (DEM)
# ==============================

with rasterio.open(dem_path) as dem:
    target_crs = dem.crs

# ==============================
# GENERIC CRS ALIGNMENT FUNCTION
# ==============================

def align_crs(gdf, name, target_crs):
    if gdf.crs is None:
        raise ValueError(f"{name} CRS is missing. Define it first.")
    
    if gdf.crs != target_crs:
        print(f"🔄 Reprojecting {name} to DEM CRS")
        return gdf.to_crs(target_crs)
    
    return gdf

# ==============================
# APPLY TO ALL VECTORS (NO REDUNDANCY)
# ==============================

vector_data = {
    "surface_flood": surface_flood,
    "river_flood": river_flood,
    "rivers": rivers,
    "roads": roads,
    "study_area": study_area
}

for name, gdf in vector_data.items():
    vector_data[name] = align_crs(gdf, name, target_crs)

# unpack back (keeps your pipeline unchanged)
surface_flood = vector_data["surface_flood"]
river_flood = vector_data["river_flood"]
rivers = vector_data["rivers"]
roads = vector_data["roads"]
study_area = vector_data["study_area"]

# ==============================
# CHECK LANDUSE CRS (RASTER)
# ==============================

with rasterio.open(landuse_path) as land:
    landuse_crs = land.crs

if landuse_crs != target_crs:
    print("⚠️ Landuse CRS differs from DEM → will reproject later")
else:
    print("✅ Landuse CRS aligned with DEM")

In [ ]:
from shapely.geometry import box, mapping
from rasterio.mask import mask
import rasterio
import os

# ==============================
# PREPARE STUDY GEOMETRY (DO ONCE)
# ==============================

study_geom = study_area.geometry.union_all()

# ==============================
# GENERIC CLIP FUNCTION (IMPORTANT)
# ==============================

def clip_raster_to_study_area(raster_path, study_geom, output_path=None):
    with rasterio.open(raster_path) as src:
        raster_bounds = src.bounds
        raster_extent = box(
            raster_bounds.left,
            raster_bounds.bottom,
            raster_bounds.right,
            raster_bounds.top
        )

        # If already inside → skip clipping
        if raster_extent.within(study_geom):
            print(f"✅ {os.path.basename(raster_path)} already within study area")
            return raster_path, src.meta.copy()

        print(f"✂️ Clipping {os.path.basename(raster_path)}")

        clipped, transform = mask(
            src,
            [mapping(study_geom)],
            crop=True
        )

        meta = src.meta.copy()
        meta.update({
            "height": clipped.shape[1],
            "width": clipped.shape[2],
            "transform": transform,
            "nodata": src.nodata
        })

    # Save only if path provided
    if output_path:
        with rasterio.open(output_path, "w", **meta) as dest:
            dest.write(clipped)
        return output_path, meta

    return clipped, meta


# ==============================
# APPLY TO DEM
# ==============================

dem_clipped_path, dem_meta = clip_raster_to_study_area(
    dem_path,
    study_geom,
    output_path="DEM_clipped.tif"
)

In [ ]:
land_clipped_path, land_meta = clip_raster_to_study_area(
    landuse_path,
    study_geom,
    output_path="landuse_clipped.tif"
)

In [ ]:
# ==============================
# GENERIC VECTOR CLIP FUNCTION
# ==============================

def clip_vector_to_study_area(gdf, study_geom, name="", output_path=None):
    
    # Optional sanity check
    if not gdf.intersects(study_geom).any():
        print(f"⚠️ Warning: No {name} overlap with study area")

    print(f"✂️ Clipping {name}...")
    
    clipped = gdf.clip(study_geom).reset_index(drop=True)

    # Save only if needed
    if output_path:
        clipped.to_file(output_path, driver="GPKG")
        print(f"💾 Saved {name} at: {output_path}")

    return clipped


# ==============================
# APPLY TO ROADS
# ==============================

roads_clipped = clip_vector_to_study_area(
    roads,
    study_geom,
    name="roads",
    output_path="roads_clipped.gpkg"
)

In [ ]:
rivers_clipped = clip_vector_to_study_area(
    rivers,
    study_geom,
    name="rivers"
)

In [ ]:
if len(rivers_clipped) == 0:
    print("⚠️ No rivers inside study area → using full rivers dataset for distance computation")
    rivers_clipped = rivers   # fallback

In [ ]:
surface_flood_clipped = clip_vector_to_study_area(
    surface_flood,
    study_geom,
    name="surface flood",
    output_path="surface_flood_clipped.gpkg"
)

river_flood_clipped = clip_vector_to_study_area(
    river_flood,
    study_geom,
    name="river/sea flood",
    output_path="river_flood_clipped.gpkg"
)

In [ ]:
from rasterio.features import rasterize
# ==============================
# RISK MAPPING (DEFINE ONCE)
# ==============================

risk_mapping = {
    "Very Low": 1,
    "Low": 2,
    "Medium": 3,
    "High": 4
}

# ==============================
# APPLY MAPPING (IN-PLACE)
# ==============================

for df, name in [
    (surface_flood_clipped, "surface"),
    (river_flood_clipped, "river")
]:
    df["risk_label"] = df["risk_band"].map(risk_mapping)
    
    if df["risk_label"].isna().any():
        raise ValueError(f"Unmapped values found in {name}_flood risk_band")

# ==============================
# LOAD DEM REFERENCE (ONCE)
# ==============================

with rasterio.open(dem_clipped_path) as dem:
    transform = dem.transform
    out_shape = (dem.height, dem.width)
    dem_crs = dem.crs

# ==============================
# GENERIC RASTERIZATION FUNCTION
# ==============================

def rasterize_flood(gdf, value_col, out_shape, transform):
    shapes = [
        (geom, value)
        for geom, value in zip(gdf.geometry, gdf[value_col])
    ]

    return rasterize(
        shapes=shapes,
        out_shape=out_shape,
        transform=transform,
        fill=0,
        dtype="uint8",
    )

# ==============================
# RASTERIZE
# ==============================

surface_raster = rasterize_flood(
    surface_flood_clipped,
    "risk_label",
    out_shape,
    transform
)

river_raster = rasterize_flood(
    river_flood_clipped,
    "risk_label",
    out_shape,
    transform
)

# ==============================
# OPTIONAL COMBINATION
# ==============================

combined_raster = np.maximum(surface_raster, river_raster)

# ==============================
# SAVE FUNCTION
# ==============================

def save_raster(array, path, meta_template):
    meta = meta_template.copy()
    meta.update({
        "count": 1,
        "dtype": "uint8",
        "nodata": 0
    })

    with rasterio.open(path, "w", **meta) as dst:
        dst.write(array, 1)

# ==============================
# SAVE OUTPUTS
# ==============================

label_meta = {
    "driver": "GTiff",
    "height": out_shape[0],
    "width": out_shape[1],
    "crs": dem_crs,
    "transform": transform
}

save_raster(surface_raster, "flood_labels_surface.tif", label_meta)
save_raster(river_raster, "flood_labels_river.tif", label_meta)
save_raster(combined_raster, "flood_labels_combined.tif", label_meta)

print("✅ Flood label rasters saved")

In [ ]:
# ==============================
# DEM + VALID MASK (KEEP THIS FOR LATER)
# ==============================

with rasterio.open(dem_clipped_path) as dem:
    dem_data = dem.read(1)
    nodata = dem.nodata
    pixel_area = dem.res[0] * dem.res[1]

valid_mask = (dem_data != nodata) if nodata is not None else ~np.isnan(dem_data)
valid_pixels = np.sum(valid_mask)

# ==============================
# FLOOD CATEGORY COUNTS
# ==============================

surface_only = np.sum(((surface_raster > 0) & (river_raster == 0)) & valid_mask)
river_only = np.sum(((surface_raster == 0) & (river_raster > 0)) & valid_mask)
both = np.sum(((surface_raster > 0) & (river_raster > 0)) & valid_mask)

total_flood = surface_only + river_only + both

# ==============================
# AREA COMPUTATION
# ==============================

valid_area_km2 = (valid_pixels * pixel_area) / 1_000_000
flood_area_km2 = (total_flood * pixel_area) / 1_000_000

# ==============================
# SAFE PERCENTAGES
# ==============================

flood_coverage = (total_flood / valid_pixels * 100) if valid_pixels > 0 else 0

surface_pct = (surface_only / total_flood * 100) if total_flood > 0 else 0
river_pct = (river_only / total_flood * 100) if total_flood > 0 else 0
both_pct = (both / total_flood * 100) if total_flood > 0 else 0

# ==============================
# OUTPUT
# ==============================

print(f"Valid study-area pixels: {valid_pixels}")
print(f"Study area (km²): {valid_area_km2:.2f}")

print(f"\nFlooded area (km²): {flood_area_km2:.2f}")
print(f"Flood coverage (%): {flood_coverage:.2f}")

print("\nBreakdown:")
print(f"Surface only (%): {surface_pct:.2f}")
print(f"River only (%): {river_pct:.2f}")
print(f"Both (%): {both_pct:.2f}")

In [ ]:
import matplotlib.pyplot as plt

labels = ["Surface Only", "River Only", "Both"]
sizes = [surface_pct, river_pct, both_pct]

explode = [0, 0.1, 0.1]

plt.figure(figsize=(6,6))
plt.pie(
    sizes,
    labels=labels,
    autopct='%1.1f%%',
    explode=explode
)

plt.title("Flood Type Composition")
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

categories = ["Study Area", "Flooded Area"]
values = [valid_area_km2, flood_area_km2]

plt.figure(figsize=(6,4))
plt.bar(categories, values)

plt.ylabel("Area (km²)")
plt.title("Study Area vs Flooded Area")

# Optional: show values on top of bars
for i, v in enumerate(values):
    plt.text(i, v, f"{v:.2f}", ha='center', va='bottom')

plt.tight_layout()
plt.show()

In [ ]:
def check_raster_alignment(ref_path, target_path, ref_name="REF", target_name="TARGET"):
    with rasterio.open(ref_path) as ref, rasterio.open(target_path) as tgt:
        
        print(f"\n--- {ref_name} vs {target_name} ---")
        
        print("CRS:", ref.crs == tgt.crs)
        print("Shape:", (ref.height, ref.width), (tgt.height, tgt.width))
        print("Transform equal:", ref.transform == tgt.transform)
        print("Resolution:", ref.res, tgt.res)

        if ref.crs != tgt.crs:
            print("⚠️ CRS mismatch")
        if (ref.height, ref.width) != (tgt.height, tgt.width):
            print("⚠️ Shape mismatch")
        if ref.transform != tgt.transform:
            print("⚠️ Transform mismatch")


check_raster_alignment(dem_clipped_path, land_clipped_path, "DEM", "Landuse")
check_raster_alignment(dem_clipped_path, sat_NDVI, "DEM", "NDVI")

In [ ]:
from rasterio.warp import reproject, Resampling
import rasterio
import numpy as np

def align_raster_to_reference(src_path, ref_path, output_path=None, resampling=Resampling.nearest):
    
    with rasterio.open(ref_path) as ref:
        ref_crs = ref.crs
        ref_transform = ref.transform
        ref_width = ref.width
        ref_height = ref.height

    with rasterio.open(src_path) as src:
        src_data = src.read(1)

        aligned = np.empty((ref_height, ref_width), dtype=src_data.dtype)

        reproject(
            source=src_data,
            destination=aligned,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=ref_transform,
            dst_crs=ref_crs,
            resampling=resampling
        )

        meta = src.meta.copy()
        meta.update({
            "height": ref_height,
            "width": ref_width,
            "transform": ref_transform,
            "crs": ref_crs,
            "nodata": src.nodata
        })

    # Save if needed
    if output_path:
        with rasterio.open(output_path, "w", **meta) as dst:
            dst.write(aligned, 1)
        print(f"✅ Saved aligned raster: {output_path}")

    return aligned, meta

In [ ]:
aligned_landuse, _ = align_raster_to_reference(
    land_clipped_path,
    dem_clipped_path,
    output_path="landuse_aligned_to_dem.tif",
    resampling=Resampling.nearest
)

In [ ]:
ndvi_aligned, _ = align_raster_to_reference(sat_NDVI, dem_clipped_path, resampling=Resampling.bilinear)
ndwi_aligned, _ = align_raster_to_reference(sat_NDWI, dem_clipped_path, resampling=Resampling.bilinear)
ndbi_aligned, _ = align_raster_to_reference(sat_NDBI, dem_clipped_path, resampling=Resampling.bilinear)

In [ ]:
check_raster_alignment(
    dem_clipped_path,
    "landuse_aligned_to_dem.tif",
    "DEM",
    "Landuse"
)

In [ ]:
from rasterio.features import rasterize
from scipy.ndimage import distance_transform_edt
import numpy as np

# ==============================
# GENERIC DISTANCE FUNCTION
# ==============================

def compute_distance_raster(
    gdf,
    dem_shape,
    dem_transform,
    valid_mask,
    name="feature",
    output_path=None
):
    # Rasterize
    binary = rasterize(
        [(geom, 1) for geom in gdf.geometry if geom is not None],
        out_shape=dem_shape,
        transform=dem_transform,
        fill=0,
        dtype="uint8"
    )

    print(f"{name} pixels:", np.sum(binary == 1))

    # Pixel size
    pixel_size_x = abs(dem_transform.a)
    pixel_size_y = abs(dem_transform.e)

    # Distance transform
    distance = distance_transform_edt(
        binary == 0,
        sampling=(pixel_size_y, pixel_size_x)
    )

    # Mask
    distance_masked = np.where(valid_mask, distance, np.nan)

    # Save if needed
    if output_path:
        meta = {
            "driver": "GTiff",
            "height": dem_shape[0],
            "width": dem_shape[1],
            "count": 1,
            "dtype": "float32",
            "crs": dem_crs,
            "transform": dem_transform,
            "nodata": np.nan
        }

        with rasterio.open(output_path, "w", **meta) as dst:
            dst.write(distance_masked.astype("float32"), 1)

        print(f"✅ Saved {name} distance raster: {output_path}")

    # Summary
    print(f"{name} distance stats:")
    print("  min:", np.nanmin(distance_masked))
    print("  max:", np.nanmax(distance_masked))
    print("  mean:", np.nanmean(distance_masked))

    return distance_masked

In [ ]:
with rasterio.open(dem_clipped_path) as dem:
    dem_data = dem.read(1)
    dem_transform = dem.transform
    dem_crs = dem.crs
    dem_shape = (dem.height, dem.width)
    dem_nodata = dem.nodata

# Valid mask
valid_mask = (dem_data != dem_nodata) if dem_nodata is not None else ~np.isnan(dem_data)

In [ ]:
road_distance = compute_distance_raster(
    roads_clipped,
    dem_shape,
    dem_transform,
    valid_mask,
    name="road",
    output_path="road_distance_to_nearest.tif"
)

In [ ]:
river_distance = compute_distance_raster(
    rivers_clipped,
    dem_shape,
    dem_transform,
    valid_mask,
    name="river",
    output_path="river_distance_to_nearest.tif"
)

In [ ]:
import os
import rasterio

def check_raster_files(file_dict, verbose=True):
    print("\n=== Checking Raster Files ===")

    readable = {}

    for name, path in file_dict.items():

        if not os.path.exists(path):
            print(f"[MISSING] {name}: {path}")
            continue

        try:
            with rasterio.open(path) as src:
                print(f"[OK] {name}")

                if verbose:
                    print(f"  CRS: {src.crs}")
                    print(f"  Shape: ({src.height}, {src.width})")
                    print(f"  Resolution: {src.res}")
                    print(f"  Nodata: {src.nodata}")

                readable[name] = path

        except Exception as e:
            print(f"[FAILED] {name}: {path}")
            print(f"  Error: {e}")

    if not readable:
        raise RuntimeError("No readable raster files found.")

    return readable

In [ ]:
satellite_files = {
    "ndvi": sat_NDVI,
    "ndwi": sat_NDWI,
    "ndbi": sat_NDBI,
}

readable_files = check_raster_files(satellite_files)

In [ ]:
from pathlib import Path
from shapely.geometry import box, mapping
import rasterio
from rasterio.mask import mask
from rasterio.warp import Resampling

# ==============================
# OUTPUT FOLDER
# ==============================
output_folder = Path("satellite_processed")
output_folder.mkdir(exist_ok=True)

# ==============================
# SAFE CLIPPING FUNCTION
# ==============================

def safe_clip_raster(raster_path, study_geom, study_crs, name, output_path):
    
    with rasterio.open(raster_path) as src:

        # Reproject study geometry if needed
        if study_crs != src.crs:
            study_geom_proj = gpd.GeoSeries([study_geom], crs=study_crs).to_crs(src.crs).iloc[0]
        else:
            study_geom_proj = study_geom

        # Check overlap
        raster_extent = box(*src.bounds)

        if not raster_extent.intersects(study_geom_proj):
            print(f"[SKIPPED] {name}: no overlap with study area")
            return None

        # Clip
        clipped, transform = mask(
            src,
            [mapping(study_geom_proj)],
            crop=True
        )

        meta = src.meta.copy()
        meta.update({
            "height": clipped.shape[1],
            "width": clipped.shape[2],
            "transform": transform
        })

        with rasterio.open(output_path, "w", **meta) as dst:
            dst.write(clipped)

        print(f"[SAVED] {name}: {output_path}")
        return output_path


# ==============================
# STEP 0: PREPARE STUDY GEOMETRY (ONLY ONCE)
# ==============================

study_geom = study_area.geometry.union_all()
study_crs = study_area.crs

# ==============================
# STEP 1: CLIP (SAFE)
# ==============================

clipped_files = {}

for feat_name, filepath in readable_files.items():

    clipped_path = safe_clip_raster(
        filepath,
        study_geom,
        study_crs,
        feat_name,
        output_folder / f"{feat_name}_clipped.tif"
    )

    if clipped_path is not None:
        clipped_files[feat_name] = clipped_path


# ==============================
# STEP 2: ALIGN TO DEM GRID
# ==============================

aligned_files = {}

for feat_name, clip_path in clipped_files.items():

    aligned_data, _ = align_raster_to_reference(
        clip_path,
        dem_clipped_path,   # ✅ FIXED
        output_path=output_folder / f"{feat_name}_aligned_to_dem.tif",
        resampling=Resampling.bilinear
    )

    aligned_files[feat_name] = aligned_data


# ==============================
# FINAL CHECK
# ==============================

print("\nFinal aligned features:")
print(list(aligned_files.keys()))

In [ ]:
sat_folder = Path("satellite_processed/")

satellite_files = {
    "ndvi": sat_folder / "ndvi_aligned_to_dem.tif",
    "ndwi": sat_folder / "ndwi_aligned_to_dem.tif",
    "ndbi": sat_folder / "ndbi_aligned_to_dem.tif",
}

# ==============================
# CHECK FILES (LESS VERBOSE)
# ==============================

readable_files = {}

for name, path in satellite_files.items():
    if not path.exists():
        print(f"[MISSING] {name}")
        continue

    try:
        with rasterio.open(path) as src:
            print(f"[OK] {name}")
            readable_files[name] = path
    except Exception as e:
        print(f"[FAILED] {name}: {e}")

if not readable_files:
    raise RuntimeError("No readable satellite rasters found.")

# ==============================
# LOAD INTO MEMORY (IMPORTANT)
# ==============================

sat_data = {}

for name, path in readable_files.items():
    with rasterio.open(path) as src:
        sat_data[name] = src.read(1)

# ==============================
# SHAPE CHECK (CRITICAL)
# ==============================

for name, arr in sat_data.items():
    if arr.shape != dem_data.shape:
        raise ValueError(f"{name} shape mismatch with DEM")

print("\nFinal satellite features loaded:", list(sat_data.keys()))

In [ ]:
from whitebox import WhiteboxTools
from pathlib import Path

wbt = WhiteboxTools()
wbt.set_verbose_mode(False)

# ==============================
# OUTPUT FOLDER
# ==============================
dem_out = Path("dem_features")
dem_out.mkdir(exist_ok=True)

# ==============================
# FILE PATHS
# ==============================
dem_input = Path(dem_clipped_path).resolve()

dem_filled = (dem_out / "dem_filled.tif").resolve()
slope_path = (dem_out / "slope.tif").resolve()
flow_acc_path = (dem_out / "flow_acc.tif").resolve()
profile_curv_path = (dem_out / "profile_curv.tif").resolve()
plan_curv_path = (dem_out / "plan_curv.tif").resolve()
depression_path = (dem_out / "depression_depth.tif").resolve()
roughness_path = (dem_out / "roughness.tif").resolve()
roughness_scale_path = (dem_out / "roughness_scale.tif").resolve()

# ==============================
# Step 1 — Fill depressions
# ==============================
if not dem_filled.exists():
    wbt.fill_depressions(
        str(dem_input),
        str(dem_filled)
    )

# Safety check
if not dem_filled.exists():
    raise FileNotFoundError("dem_filled.tif was not created. Stop here.")

# ==============================
# Step 2 — Slope
# ==============================
if not slope_path.exists():
    wbt.slope(
        str(dem_filled),
        str(slope_path),
        units="degrees"
    )

# ==============================
# Step 3 — Flow accumulation
# ==============================
if not flow_acc_path.exists():
    wbt.d8_flow_accumulation(
        str(dem_filled),
        str(flow_acc_path),
        out_type="cells"
    )

# ==============================
# Step 4 — Curvatures
# ==============================
if not profile_curv_path.exists():
    wbt.profile_curvature(
        str(dem_filled),
        str(profile_curv_path)
    )

if not plan_curv_path.exists():
    wbt.plan_curvature(
        str(dem_filled),
        str(plan_curv_path)
    )

# ==============================
# Step 5 — Depression depth
# ==============================
if not depression_path.exists():
    wbt.subtract(
        str(dem_filled),
        str(dem_input),
        str(depression_path)
    )

# ==============================
# Step 6 — Roughness
# ==============================
if not roughness_path.exists():
    wbt.multiscale_roughness(
        str(dem_input),
        str(roughness_path),
        str(roughness_scale_path),
        min_scale=1,
        max_scale=3
    )

# ==============================
# FINAL CHECK
# ==============================
print("\nFile existence check:")
print("DEM filled:", dem_filled.exists())
print("Slope:", slope_path.exists())
print("Flow acc:", flow_acc_path.exists())
print("Profile curvature:", profile_curv_path.exists())
print("Plan curvature:", plan_curv_path.exists())
print("Depression depth:", depression_path.exists())
print("Roughness:", roughness_path.exists())

print("\n✅ DEM feature generation completed.")

In [ ]:
import os

print("dem_filled path:", dem_filled)
print("Exists?", os.path.exists(dem_filled))


import os

print("Current working dir:", os.getcwd())
print("Whitebox working dir:", wbt.get_working_dir())

In [ ]:
from scipy.ndimage import generic_filter

# Use correct path variable (DO NOT hardcode string)
with rasterio.open(dem_filled) as src:
    dem_filled_data = src.read(1).astype("float32")

# Use existing valid_mask (DO NOT redefine)
dem_filled_data = np.where(valid_mask, dem_filled_data, np.nan)

# Local std (3x3)
local_std = generic_filter(dem_filled_data, np.nanstd, size=3)
local_std = np.where(valid_mask, local_std, np.nan).astype("float32")

In [ ]:
local_std_path = dem_out / "local_std.tif"

with rasterio.open(dem_filled) as src:
    meta = src.meta.copy()

meta.update({
    "dtype": "float32",
    "nodata": np.nan
})

with rasterio.open(local_std_path, "w", **meta) as dst:
    dst.write(local_std, 1)

print("Saved local_std:", local_std_path)

In [ ]:
# DEM (already loaded earlier)
dem = dem_data.astype("float32")

# valid_mask already defined earlier — DO NOT redefine

# Correct save function (use ONLY this)
def save_raster(array, ref_path, out_path):
    with rasterio.open(ref_path) as src:
        meta = src.meta.copy()

    meta.update(dtype="float32", count=1, nodata=np.nan)

    with rasterio.open(out_path, "w", **meta) as dst:
        dst.write(array.astype("float32"), 1)

In [ ]:
from scipy.ndimage import generic_filter

def percentile_func(window):
    center = window[len(window)//2]
    valid = window[~np.isnan(window)]
    
    if len(valid) == 0:
        return np.nan
    
    return np.sum(valid < center) / len(valid)

elev_percentile = generic_filter(dem, percentile_func, size=7)

elev_percentile = np.where(valid_mask, elev_percentile, np.nan)

save_raster(
    elev_percentile,
    dem_clipped_path,
    dem_out / "elev_percentile.tif"
)

In [ ]:
from scipy.ndimage import uniform_filter

mean = uniform_filter(dem, size=5)
mean_sq = uniform_filter(dem**2, size=5)

elev_var = mean_sq - mean**2
elev_var = np.maximum(elev_var, 0)

elev_std = np.sqrt(elev_var)

elev_contrast = elev_std / (np.abs(mean) + 1e-6)

elev_contrast = np.where(valid_mask, elev_contrast, np.nan)

save_raster(
    elev_contrast,
    dem_clipped_path,
    dem_out / "elev_contrast.tif"
)

In [ ]:
from scipy.ndimage import uniform_filter
import rasterio
import numpy as np

with rasterio.open("dem_features/slope.tif") as src:
    slope = src.read(1).astype("float32")
    slope_nodata = src.nodata

# Handle nodata
if slope_nodata is not None:
    slope = np.where(slope == slope_nodata, np.nan, slope)

# Fill NaNs BEFORE filtering
slope_filled = np.nan_to_num(slope, nan=0.0)

# Compute stats
mean_s = uniform_filter(slope_filled, size=5)
mean_s_sq = uniform_filter(slope_filled**2, size=5)

var_s = mean_s_sq - mean_s**2
var_s = np.maximum(var_s, 0)

slope_var = np.sqrt(var_s)

# Restore NaN mask
slope_var = np.where(np.isnan(slope), np.nan, slope_var)

# Save
with rasterio.open("DEM_clipped.tif") as ref:
    meta = ref.meta.copy()

meta.update(dtype="float32", nodata=np.nan)

with rasterio.open("dem_features/slope_variability.tif", "w", **meta) as dst:
    dst.write(slope_var.astype("float32"), 1)

print("✔ slope_variability fixed")

In [ ]:
from scipy.ndimage import uniform_filter

with rasterio.open("dem_features/flow_acc.tif") as src:
    flow = src.read(1).astype("float32")
    flow_nodata = src.nodata

# Handle nodata
if flow_nodata is not None:
    flow = np.where(flow == flow_nodata, np.nan, flow)

# Log transform
flow = np.where(flow < 0, 0, flow)
flow_log = np.log1p(flow)

# Fill NaNs BEFORE filtering
flow_filled = np.nan_to_num(flow_log, nan=0.0)

# Compute local mean
mean_f = uniform_filter(flow_filled, size=5)

flow_contrast = flow_log - mean_f

# Restore NaNs
flow_contrast = np.where(np.isnan(flow_log), np.nan, flow_contrast)

# Save
with rasterio.open("DEM_clipped.tif") as ref:
    meta = ref.meta.copy()

meta.update(dtype="float32", nodata=np.nan)

with rasterio.open("dem_features/flow_contrast.tif", "w", **meta) as dst:
    dst.write(flow_contrast.astype("float32"), 1)

print("✔ flow_contrast fixed")

In [ ]:
# ==============================
# Distance to river (already computed)
# ==============================

# Ensure float
river_dist = river_distance.astype("float32")

# Apply mask
river_dist = np.where(valid_mask, river_dist, np.nan)

# Normalize
river_dist_norm = river_dist / np.nanmax(river_dist)

# Save
save_raster(
    river_dist_norm,
    dem_clipped_path,
    dem_out / "river_distance_norm.tif"
)

In [ ]:
# ==============================
# Distance to road (already computed)
# ==============================

road_dist = road_distance.astype("float32")

# Apply mask
road_dist = np.where(valid_mask, road_dist, np.nan)

# Normalize
road_dist_norm = road_dist / np.nanmax(road_dist)

# Save
save_raster(
    road_dist_norm,
    dem_clipped_path,
    dem_out / "road_distance_norm.tif"
)

In [ ]:
import rasterio
import numpy as np

# -----------------------------------
# Step 1 — Helper function
# -----------------------------------
def read_raster(path):
    with rasterio.open(path) as src:
        data = src.read(1).astype("float32")
        nodata = src.nodata
    return data, nodata

# -----------------------------------
# Step 2 — Load rasters
# -----------------------------------
elevation, dem_nodata = read_raster("DEM_clipped.tif")
slope, _ = read_raster("dem_features/slope.tif")
flow_acc, _ = read_raster("dem_features/flow_acc.tif")
profile_curv, _ = read_raster("dem_features/profile_curv.tif")
plan_curv, _ = read_raster("dem_features/plan_curv.tif")
depression_depth, _ = read_raster("dem_features/depression_depth.tif")
local_std, _ = read_raster("dem_features/local_std.tif")

# ---- NEW FEATURES ----
elev_percentile, _ = read_raster("dem_features/elev_percentile.tif")
elev_contrast, _ = read_raster("dem_features/elev_contrast.tif")
slope_var, _ = read_raster("dem_features/slope_variability.tif")
flow_contrast, _ = read_raster("dem_features/flow_contrast.tif")

# ---- Distance ----
road_dist, _ = read_raster("dem_features/road_distance_norm.tif")
river_dist, _ = read_raster("dem_features/river_distance_norm.tif")

# ---- Landuse ----
landuse, landuse_nodata = read_raster("landuse_aligned_to_dem.tif")

# ---- Labels ----
labels, _ = read_raster("flood_labels_surface.tif")

# ---- Satellite ----
ndvi, _ = read_raster("satellite_processed/ndvi_aligned_to_dem.tif")
ndwi, _ = read_raster("satellite_processed/ndwi_aligned_to_dem.tif")
ndbi, _ = read_raster("satellite_processed/ndbi_aligned_to_dem.tif")

# -----------------------------------
# Step 3 — Preprocess
# -----------------------------------
flow_acc = np.where(flow_acc < 0, 0, flow_acc)
flow_acc = np.log1p(flow_acc)

# -----------------------------------
# Step 4 — Create VALID MASK (FIXED)
# -----------------------------------

# ONLY core features here
if dem_nodata is not None:
    valid_mask = elevation != dem_nodata
else:
    valid_mask = ~np.isnan(elevation)

valid_mask &= ~np.isnan(slope)
valid_mask &= ~np.isnan(flow_acc)
valid_mask &= ~np.isnan(profile_curv)
valid_mask &= ~np.isnan(plan_curv)
valid_mask &= ~np.isnan(depression_depth)
valid_mask &= ~np.isnan(local_std)

# Landuse
if landuse_nodata is not None:
    valid_mask &= landuse != landuse_nodata
else:
    valid_mask &= landuse > 0

# Labels
valid_mask &= labels >= 0
valid_mask &= ~np.isnan(labels)

print("Valid pixels after core mask:", np.sum(valid_mask))

np.save("valid_mask_1.npy", valid_mask)
print("valid_mask saved:", valid_mask.shape)


# -----------------------------------
# Step 4.1 — Fill OPTIONAL features
# -----------------------------------

def safe_fill(arr):
    return np.where(np.isnan(arr), np.nanmean(arr), arr)

elev_percentile = safe_fill(elev_percentile)
elev_contrast = safe_fill(elev_contrast)
slope_var = safe_fill(slope_var)
flow_contrast = safe_fill(flow_contrast)

road_dist = safe_fill(road_dist)
river_dist = safe_fill(river_dist)

ndvi = safe_fill(ndvi)
ndwi = safe_fill(ndwi)
ndbi = safe_fill(ndbi)

# -----------------------------------
# Step 5 — Landuse encoding
# -----------------------------------
from sklearn.preprocessing import OneHotEncoder

landuse_valid = landuse[valid_mask].reshape(-1, 1)

encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
landuse_encoded = encoder.fit_transform(landuse_valid)

print("Landuse encoded shape:", landuse_encoded.shape)

# -----------------------------------
# Step 6 — Stack features
# -----------------------------------

X_continuous = np.stack([
    elevation[valid_mask],
    slope[valid_mask],
    flow_acc[valid_mask],
    profile_curv[valid_mask],
    plan_curv[valid_mask],
    depression_depth[valid_mask],
    local_std[valid_mask],

    # NEW
    elev_percentile[valid_mask],
    elev_contrast[valid_mask],
    slope_var[valid_mask],
    flow_contrast[valid_mask],

    # Distance
    road_dist[valid_mask],
    river_dist[valid_mask],

    # Satellite
    ndvi[valid_mask],
    ndwi[valid_mask],
    ndbi[valid_mask]

], axis=1)

X = np.hstack([X_continuous, landuse_encoded])
y = labels[valid_mask].astype("int32")

# -----------------------------------
# Step 7 — Validation
# -----------------------------------
print("X_continuous shape:", X_continuous.shape)
print("X final shape:", X.shape)
print("y shape:", y.shape)

print("Any NaN in X:", np.isnan(X).any())
print("Any NaN in y:", np.isnan(y).any())

print("Number of features:", X.shape[1])
print("Unique labels:", np.unique(y))

# -----------------------------------
# Step 8 — Save
# -----------------------------------
np.save("X.npy", X)
np.save("y.npy", y)

print("Dataset saved successfully")

In [ ]:
feature_names_cont = [
    "elevation",
    "slope",
    "flow_acc",
    "profile_curv",
    "plan_curv",
    "depression_depth",
    "local_std",
    "elev_percentile",
    "elev_contrast",
    "slope_variability",
    "flow_contrast",
    "road_dist",
    "river_dist",
    "ndvi",
    "ndwi",
    "ndbi"
]

for i, name in enumerate(feature_names_cont):
    col = X_continuous[:, i]
    print(
        name,
        "NaN:", np.isnan(col).sum(),
        "Inf:", np.isinf(col).sum(),
        "Min:", np.nanmin(col),
        "Max:", np.nanmax(col)
    )